# Challenge:

- Build a synthetic dataset generator, write models that can generate datasets.
- Use a variety of models and prompts for diverse output, try quantizing and not quantizing, try different shapes and sizes.
- Create a Gradio UI for your product

## Idea

- Generate sample data for use when developing a REST API.

### Input: API spec, in different formats.

(API spec is already known vs suggest structure and endpoints based on model?)
  
- Empty JSON structure
- Java
- OAD (OpenAPI document) - in JSON or YAML
- UML? (Image or Mermaid chart def)
- URL - for example running Swagger UI instance
- As GUI - dropdowns or other UI elements to select entities and relations

In [10]:
# Imports

from transformers import (
    pipeline,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
import os
from dotenv import load_dotenv
from openai import OpenAI
from huggingface_hub import login

load_dotenv(override=True)

True

In [ ]:
# Constants

LLAMA_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"
QWEN_MODEL = "Qwen/Qwen2.5-Coder-32B-Instruct"
PHI_MODEL = "microsoft/phi-2.5-coder"
OPENAPI_FILE = "storefront-sample.json"

In [ ]:
# Huggingface login

login(token=os.getenv("HF_TOKEN"), add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
# in case $ref is used in the schema - resolve_ref


def resolve_ref(spec, ref):
    path = ref.replace("#/", "").split("/")
    obj = spec
    for p in path:
        obj = obj[p]
    return obj

In [ ]:
# Extraction function

import json


def extract_schema(openapi, path, method):
    method = method.lower()
    endpoint = openapi["paths"][path][method]
    schema = endpoint["requestBody"]["content"]["application/json"]["schema"]
    return schema


with open("OPENAPI_FILE") as f:
    spec = json.load(f)

schema = extract_schema(spec, "/cart/items", "post")

if "$ref" in schema:
    schema = resolve_ref(spec, schema["$ref"])

print(json.dumps(schema, indent=2))

{
  "type": "object",
  "required": [
    "product_id",
    "quantity"
  ],
  "properties": {
    "product_id": {
      "type": "string",
      "format": "uuid"
    },
    "quantity": {
      "type": "integer",
      "minimum": 1
    }
  }
}


In [ ]:
user_prompt = f"""Generate realistic JSON objects following this schema:
{schema}
"""
system_prompt = """
You excel at generating realistic JSON objects following a given schema.
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [ ]:
# Create pipeline

task = "text-generation"
model = "gpt2"
device = 0

json_data_generator = pipeline(task, model, device)

result = json_data_generator(spec)

In [ ]:
# Validate JSON

import json


def validate_json(output):
    try:
        data = json.loads(output)
        return True
    except:
        return False